# 🌌 PolyMerge: Universal Architecture-Agnostic LLM Merger & Toolkit

Interactive Production Notebook for **Google Colab** and **Kaggle**.

### ⚡ Key Capabilities:
1. **Core Model Merging**: 12 Methods (`linear`, `slerp`, `nuslerp`, `karcher`, `nearswap`, `task_arithmetic`, `ties`, `dare_ties`, `della`, `omnimerge_v2`, `model_stock`, `sce`).
2. **Vision / Multimodal Auto-Filter**: Automatically isolates text backbone and drops vision layers (`visual.*`, `vision_tower.*`, `mm_projector.*`).
3. **Zero-Class Streaming**: Directly manipulates raw Safetensors and GGUF files with lowest RAM consumption and sequential cache eviction.
4. **MoE Upcycling**: Dense-to-MoE assembly (`Mixtral`, `Qwen-MoE`, `DeepSeek-MoE`, `Gemma 4 MoE`).
5. **Layer Slicing & Frankenmerging**: Arbitrary layer duplication and stacking (e.g. 32L -> 48L).
6. **Model Pruning**: Zero-retraining Dense Layer Pruning (Local Residual Healing) & MoE Expert Pruning (T18 Soft-Transfer).
7. **LoRA Toolkit & Tokenizer Surgery**: Truncated SVD extraction, permanent base baking, vocabulary alignment, and truncation.
8. **Sanity Audit & Direct HF Push**: Check NaN/Inf parameters, router condition numbers, and publish straight to Hugging Face Hub.

---  
## 1. 📦 Installation & Setup

In [ ]:
#@title 1. Install Dependencies & Clone PolyMerge
!pip install -q pyyaml rich huggingface-hub safetensors hf-transfer
!rm -rf polymerge
!git clone https://github.com/ehristoforu/polymerge.git
%cd polymerge
!pip install -q -e .
print("✓ PolyMerge successfully installed!")

---
## 1.1. 💾 Google Drive Storage Mount (Unlimited Output Capacity)
Mount Google Drive to persist large model outputs, prevent local Colab disk OOM, and automatically organize merges in timestamped directories ().

In [ ]:
#@title 1.1. Mount Google Drive & Initialize Timestamped Workspace
use_google_drive = True #@param {type:"boolean"}

import os
from datetime import datetime
from pathlib import Path

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

if use_google_drive:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        workspace_root = Path(f"/content/drive/MyDrive/polymerge_{timestamp}")
        workspace_root.mkdir(parents=True, exist_ok=True)
        default_output_dir = str(workspace_root / "output_model")
        print(f"✓ Google Drive connected!")
        print(f"✓ Session Workspace initialized at: {workspace_root}")
        print(f"✓ Default Model Output Dir: {default_output_dir}")
    except Exception as e:
        print(f"⚠️ Failed to mount Google Drive ({e}). Falling back to local Colab storage.")
        workspace_root = Path(f"./polymerge_{timestamp}")
        workspace_root.mkdir(parents=True, exist_ok=True)
        default_output_dir = str(workspace_root / "output_model")
else:
    workspace_root = Path(f"./polymerge_{timestamp}")
    workspace_root.mkdir(parents=True, exist_ok=True)
    default_output_dir = str(workspace_root / "output_model")
    print(f"ℹ️ Using local disk workspace: {workspace_root}")


---  
## 2. ⚙️ Hardware Platform Configuration

In [ ]:
#@title 2. Select Hardware Platform & Optimization Tuning
platform = "gpu" #@param ["cpu", "cpu+highram", "gpu", "tpu"]
chunk_size_mb = 2000 #@param {type:"integer"}

import os
import torch

os.environ["POLYMERGE_DEVICE"] = "auto"
chunk_bytes = chunk_size_mb * 1_000_000

if platform == "cpu":
    os.environ["POLYMERGE_DEVICE"] = "cpu"
    print("⚙️ Platform: CPU Standard. Chunking set to:", chunk_size_mb, "MB")
elif platform == "cpu+highram":
    os.environ["POLYMERGE_DEVICE"] = "cpu"
    chunk_bytes = 4_000_000_000
    print("⚙️ Platform: CPU High-RAM. Shard chunk size expanded to 4GB.")
elif platform == "gpu":
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        os.environ["POLYMERGE_DEVICE"] = "cuda"
        print(f"⚙️ Platform: GPU ({gpu_name}, {vram_gb:.1f} GB VRAM). Hardware acceleration ACTIVE.")
    else:
        print("⚠️ GPU not detected! Falling back to CPU. (Change Runtime type to T4 GPU)")
        os.environ["POLYMERGE_DEVICE"] = "cpu"
elif platform == "tpu":
    try:
        import torch_xla.core.xla_model as xm
        os.environ["POLYMERGE_DEVICE"] = "tpu"
        print("⚙️ Platform: Cloud TPU detected! Hardware routing via PyTorch/XLA.")
    except ImportError:
        print("⚠️ PyTorch/XLA not found. Installing torch_xla for TPU support...")
        !pip install -q torch_xla
        os.environ["POLYMERGE_DEVICE"] = "tpu"
        print("⚙️ TPU support installed.")

---  
## 3. 🔑 Secrets & Hugging Face High-Speed Transfer

In [ ]:
#@title 3. Hugging Face Login (Secrets & Fast Transfer)
import os
from huggingface_hub import login

# Enable high-speed Rust-based multi-threaded transfer
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print("✓ Found HF_TOKEN in Colab Secrets.")
except Exception:
    pass

if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    print("✓ Successfully authenticated with Hugging Face!")
else:
    print("ℹ️ HF_TOKEN not set in Colab Secrets. You can enter it manually below if accessing gated/private models:")
    # login() # Uncomment to login interactively

---  
## 4. 📝 Recipe Library & Configurations
Select and run any of the specialized task recipes below to activate its configuration.

In [ ]:
#@title 4.1. Recipe: DARE-TIES Weight Consensus Merge
yaml_config = """
merge_method: dare_ties
base_model: meta-llama/Llama-3.2-1B
models:
  - model: meta-llama/Llama-3.2-1B-Instruct
    parameters:
      weight: 0.6
      density: 0.5
  - model: meta-llama/Llama-3.2-1B
    parameters:
      weight: 0.4
      density: 0.5
parameters:
  normalize: true
dtype: bfloat16
ignore_vision: true
"""
merge_task_mode = "merge"
with open("active_config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config.strip())
print("✓ DARE-TIES recipe active! Saved to active_config.yaml")

In [ ]:
#@title 4.2. Recipe: OmniMerge v2 (OBIM-lite + DAREx + EMR Sign Election)
yaml_config = """
merge_method: omnimerge_v2
base_model: Qwen/Qwen2.5-7B
models:
  - model: Qwen/Qwen2.5-Coder-7B-Instruct
    weight: 0.5
    density: 0.6
  - model: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
    weight: 0.5
    density: 0.6
parameters:
  darex_q: 0.75
  features: ["obim", "darex", "emr"]
dtype: bfloat16
ignore_vision: true
skip_patterns:
  - "mlp.gate_proj"
  - "mlp.up_proj"
  - "mlp.down_proj"
"""
merge_task_mode = "merge"
with open("active_config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config.strip())
print("✓ OmniMerge v2 recipe active! Saved to active_config.yaml")

In [ ]:
#@title 4.3. Recipe: Dense-to-MoE Upcycling (Qwen / Mixtral / DeepSeek / Gemma)
yaml_config = """
base_model: mistralai/Mistral-7B-v0.3
models:
  - mistralai/Mistral-7B-Instruct-v0.3
  - Qwen/Qwen2.5-Coder-7B-Instruct
  - deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
target_format: mixtral
gate_mode: uniform_random
shared_expert: mistralai/Mistral-7B-v0.3
dtype: bfloat16
"""
merge_task_mode = "moe"
with open("active_config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config.strip())
print("✓ MoE Upcycling recipe active! Saved to active_config.yaml (Mode: --moe)")

In [ ]:
#@title 4.4. Recipe: Layer Slicing & Frankenmerging (e.g. 32L -> 48L Stacking)
yaml_config = """
slices:
  - sources:
      - model: meta-llama/Llama-3.2-1B
        layer_range: [0, 12]
  - sources:
      - model: meta-llama/Llama-3.2-1B-Instruct
        layer_range: [6, 16]
"""
merge_task_mode = "slices"
with open("active_config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config.strip())
print("✓ Layer Slicing recipe active! Saved to active_config.yaml (Mode: --slices)")

In [ ]:
#@title 4.5. Recipe: Zero-Retraining Model Pruning (Dense & MoE Experts)
# Set parameters for direct pruning execution
prune_mode = "moe" #@param ["dense", "moe"]
prune_target_model = "google/gemma-4-26B-A4B-it" #@param {type:"string"}
prune_dense_fraction = 0.2 #@param {type:"number"}
prune_moe_target_experts = 64 #@param {type:"integer"}
merge_task_mode = f"prune_{prune_mode}"
print(f"✓ Pruning task configured for: {prune_mode.upper()} on {prune_target_model}")

In [ ]:
#@title 4.6. Recipe: LoRA Toolkit (SVD Extraction & In-Place Base Baking)
lora_action = "extract" #@param ["extract", "bake"]
lora_base = "meta-llama/Llama-3.2-1B" #@param {type:"string"}
lora_tuned = "meta-llama/Llama-3.2-1B-Instruct" #@param {type:"string"}
lora_adapter_path = "./extracted_lora" #@param {type:"string"}
lora_rank = 16 #@param {type:"integer"}
lora_alpha = 32.0 #@param {type:"number"}
merge_task_mode = f"lora_{lora_action}"
print(f"✓ LoRA {lora_action.upper()} configured!")

---  
## 5. 🔍 Resource Audit & Environment Validation

In [ ]:
#@title 5. Audit Hardware Resources & Environment Health
import psutil
import torch
from polymerge.utils import human_bytes

ram = psutil.virtual_memory()
disk = psutil.disk_usage('/')

print("=================== ENVIRONMENT AUDIT ===================")
print(f"• System RAM Available : {human_bytes(ram.available)} / {human_bytes(ram.total)}")
print(f"• Disk Space Free      : {human_bytes(disk.free)} / {human_bytes(disk.total)}")
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory
    print(f"• GPU Accelerator      : {torch.cuda.get_device_name(0)} ({human_bytes(vram)} VRAM)")
else:
    print("• Compute Device       : CPU Mode")
print("• Selected Target Device:", os.environ.get("POLYMERGE_DEVICE", "auto"))
print(f"• Active Task Mode     : {merge_task_mode}")
print("=========================================================")

---  
## 6. 🚀 Launch Task Execution (Merge / Upcycle / Prune / LoRA)

In [ ]:
#@title 6. Execute Selected PolyMerge Task
output_directory = default_output_dir if "default_output_dir" in locals() else "./merged_model" #@param {type:"string"}

if merge_task_mode == "merge":
    print("Running Standard / DARE / OmniMerge Pipeline...")
    !polymerge active_config.yaml -o {output_directory} --chunk-size {chunk_bytes}
elif merge_task_mode == "moe":
    print("Running Dense-to-MoE Upcycling...")
    !polymerge active_config.yaml -o {output_directory} --moe --chunk-size {chunk_bytes}
elif merge_task_mode == "slices":
    print("Running Layer Slicing & Frankenmerging...")
    !polymerge active_config.yaml -o {output_directory} --slices --chunk-size {chunk_bytes}
elif merge_task_mode == "prune_dense":
    print("Running Dense Layer Pruning with Local Healing...")
    !polymerge --prune-dense --base {prune_target_model} -o {output_directory} --drop-fraction {prune_dense_fraction}
elif merge_task_mode == "prune_moe":
    print("Running MoE Expert Pruning with T18 Soft-Transfer Router Recovery...")
    !polymerge --prune-moe --base {prune_target_model} -o {output_directory} --target-experts {prune_moe_target_experts} --alpha 0.3
elif merge_task_mode == "lora_extract":
    print("Running LoRA SVD Extraction...")
    !polymerge --extract-lora --base {lora_base} --tuned {lora_tuned} -o {output_directory} --rank {lora_rank} --alpha {lora_alpha}
elif merge_task_mode == "lora_bake":
    print("Baking LoRA permanently into Base Model...")
    !polymerge --merge-lora --base {lora_base} --adapter {lora_adapter_path} -o {output_directory}

# In-flight model sanity audit
if os.path.exists(output_directory):
    print("\nRunning in-flight model sanity audit...")
    !polymerge --audit --base {output_directory}

---  
## 7. 📤 Direct Publishing to Hugging Face Hub

In [ ]:
#@title 7. Publish Result to Hugging Face Hub
hub_repo_id = "username/my-merged-model" #@param {type:"string"}
make_private = False #@param {type:"boolean"}
commit_msg = "Upload merged model via PolyMerge Colab" #@param {type:"string"}

import os
from pathlib import Path
from polymerge.publisher import PublisherTool

# Automatically check and use the output directory from the previous cell
target_dir = output_directory if 'output_directory' in locals() and Path(output_directory).exists() else "./merged_model"

if hub_repo_id == "username/my-merged-model" or not hub_repo_id.strip():
    print("⚠️ Please provide a valid Hugging Face repo ID (e.g. 'my-username/my-awesome-merge')!")
else:
    token_to_use = hf_token if 'hf_token' in locals() and hf_token else None
    print(f"Publishing model from '{target_dir}' to https://huggingface.co/{hub_repo_id}...")
    PublisherTool.push_to_hub(
        model_dir=target_dir,
        repo_id=hub_repo_id.strip(),
        private=make_private,
        token=token_to_use,
        commit_message=commit_msg,
    )
    print(f"\n✓ Model successfully published to: https://huggingface.co/{hub_repo_id}")